<a href="https://colab.research.google.com/github/latikasekhri6577/Langchain-LangGraph-Course/blob/main/Langchain_RAG_Course.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Install Packages**

Installs the LangChain libraries into Colab. You must run it again after every session restart.

In [1]:
!pip install -q langchain langchain-openai

In [2]:
!pip install -q langchain langchain-google-genai pydantic

**2.Load the Key and Create the Model**

Reads your secret key and creates the model object, which is your connection to the LLM. Swapping providers only changes this cell.

In [3]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

**3. Call the LLM**

model.invoke("...") sends text to the model and returns a message object. .content is the answer text.

In [4]:
response = model.invoke("Explain what an API is in 2 lines.")
print(response.content)

An API (Application Programming Interface) is a set of definitions and protocols that allows different software applications to communicate. It acts as an intermediary, enabling programs to request services or exchange data with each other.


**4. Prompt Templates**

A reusable prompt with blanks like {topic} that you fill in each time. The system message sets the model's role, and the human message holds the question.

In [5]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly teacher who explains things simply."),
    ("human", "Explain {topic} to a {level} in 3 lines."),
])

messages = prompt.invoke({"topic": "recursion", "level": "beginner"})
print(messages)

response = model.invoke(messages)
print(response.content)

messages=[SystemMessage(content='You are a friendly teacher who explains things simply.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain recursion to a beginner in 3 lines.', additional_kwargs={}, response_metadata={})]
Imagine a function that solves a problem by calling itself.
It keeps breaking the problem into smaller, identical pieces until it reaches a simple base case.
Crucially, it needs a "stop" condition, or it will repeat forever!


**5. Parsing the output**

StrOutputParser pulls the plain text out of the model's message object. This gives you a clean string to use in later steps.

In [6]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

text = parser.invoke(response)
print(text)
print(type(text))

Imagine a function that solves a problem by calling itself.
It keeps breaking the problem into smaller, identical pieces until it reaches a simple base case.
Crucially, it needs a "stop" condition, or it will repeat forever!
<class 'langchain_core.messages.base.TextAccessor'>


**6. Simple chain**

prompt | model | parser joins the three pieces with a pipe, so one invoke call runs all of them in order.

In [7]:
chain = prompt | model | parser

answer = chain.invoke({"topic": "recursion", "level": "beginner"})
print(answer)

Imagine a function that solves a problem by calling itself.
It keeps breaking the problem into smaller, identical pieces until it reaches a simple base case.
Crucially, it needs a "stop" condition, or it will repeat forever!


**7. Structured Output**

You define a Pydantic class and the model returns an object of that shape. You can then read result.name directly instead of parsing a paragraph.

In [8]:
from pydantic import BaseModel, Field
from typing import List

class Concept(BaseModel):
    name: str = Field(description="Name of the concept")
    definition: str = Field(description="One-line definition")
    examples: List[str] = Field(description="Two short examples")
    difficulty: str = Field(description="easy, medium or hard")

structured_model = model.with_structured_output(Concept)

result = structured_model.invoke("Tell me about recursion.")
print(result)
print(result.name)
print(result.examples[0])

name='Recursion' definition='A programming technique where a function calls itself to solve a problem by breaking it down into smaller, similar subproblems.' examples=['Calculating the factorial of a number (e.g., factorial(n) = n * factorial(n-1)).', 'Traversing a tree data structure (e.g., visiting a node, then recursively visiting its children).'] difficulty='medium'
Recursion
Calculating the factorial of a number (e.g., factorial(n) = n * factorial(n-1)).


# **Document Processing in RAG (First Stage)**

**1.Install Packages**

In [9]:
!pip install -q langchain-community langchain-text-splitters pypdf docx2txt beautifulsoup4

In [10]:
from google.colab import files
uploaded = files.upload()

**Understand the Document object**

Everything in LangChain RAG is a Document with two parts: the text and its metadata.

page_content is the text. metadata records where it came from, which is how a RAG bot can later cite its sources.

In [11]:
from langchain_core.documents import Document

d = Document(page_content="A foreign key links two tables.", metadata={"source": "dbms.pdf", "page": 3})
print(d.page_content)
print(d.metadata)

A foreign key links two tables.
{'source': 'dbms.pdf', 'page': 3}


**2. Load a PDF**

Each page becomes one Document, and the metadata holds the file name and page number.

In [12]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Operating System Notes.pdf")   # change to your file name
docs = loader.load()

print(len(docs))                  # one Document per page
print(docs[0].page_content[:100])
print(type(docs[0]))
print(docs[0].metadata)

/tmp/ipykernel_36612/2573522691.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


12
Operating Systems
●
An
Operating
System
can
be
defined
as
an
interface
between
user
and
hardware
.
I
<class 'langchain_core.documents.base.Document'>
{'producer': 'Skia/PDF m93 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Operating System Notes', 'source': '/content/Operating System Notes.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}


**Other loaders**

Run only the lines for files you have. The pattern is always the same: pick the loader for the file type, call .load(), get a list of Document objects. The rest of your RAG code doesn't care where they came from.

In [13]:
from langchain_community.document_loaders import TextLoader, Docx2txtLoader, WebBaseLoader

txt_docs  = TextLoader("/content/simpletext.txt").load()
docx_docs = Docx2txtLoader("/content/MiniProjectReport.docx").load()
web_docs  = WebBaseLoader("https://en.wikipedia.org/wiki/Relational_database").load()

print(len(txt_docs), len(docx_docs), len(web_docs))

1 1 1


**Split into chunks**

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = splitter.split_documents(docs)

print(len(docs), "pages ->", len(chunks), "chunks")
print(chunks[0].page_content)
print(chunks[0].metadata)

print(chunks[-1].page_content)
print(chunks[-1].metadata)

12 pages -> 25 chunks
Operating Systems
●
An
Operating
System
can
be
defined
as
an
interface
between
user
and
hardware
.
It
is
responsible
for
the
execution
of
all
the
processes,
Resource
Allocation,
CPU
management,
File
Management
and
many
other
tasks.
The
purpose
of
an
operating
system
is
to
provide
an
environment
in
which
a
user
can
execute
programs
in
a
convenient and efficient manner.
●
Types of Operating Systems
:
1.
Batch OS
–
A set of similar jobs are stored in the
main memory for execution. A job gets
assigned to the CPU, only when the execution of the
previous job completes.
2.
Multiprogramming OS
–
The main memory consists of
jobs waiting for CPU time. The
OS selects one of the processes and assigns it to
the CPU. Whenever the executing
process needs to wait for any other operation (like
I/O), the OS selects another process
from the job queue and assigns it to the CPU.
This
way, the CPU is never kept idle
and the user gets the flavor of getting multiple tasks
done at once.
3

chunk_size is the maximum characters per chunk.

chunk_overlap repeats the end of one chunk at the start of the next, so an idea isn't cut in half.

RecursiveCharacterTextSplitter tries to split at paragraphs first, then lines, then words, so chunks stay readable.

The metadata is kept on every chunk.

**Why chunking matters?**

A page can hold several topics, and the model's context window is limited. A search for "foreign key" should return the paragraph about foreign keys, not a whole page of mixed content. So we split into small overlapping chunks.

**Experiment with sizes**

Smaller chunks mean more of them and more precise retrieval, but each carries less context. Larger chunks keep context but retrieve less precisely.

In [15]:
for size, overlap in [(300, 50), (1000, 200), (2000, 400)]:
    s = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    c = s.split_documents(docs)
    print(f"size={size}, overlap={overlap} -> {len(c)} chunks")

size=300, overlap=50 -> 76 chunks
size=1000, overlap=200 -> 25 chunks
size=2000, overlap=400 -> 12 chunks


**Inspect your chunks**

Always read a few chunks before building anything on top of them. If chunks look broken or messy, your retrieval will be poor, whatever model you use.

In [16]:
for i in [0, 5, 10]:
    print(f"--- chunk {i} | page {chunks[i].metadata.get('page')} ---")
    print(chunks[i].page_content[:300])
    print()

--- chunk 0 | page 0 ---
Operating Systems
●
An
Operating
System
can
be
defined
as
an
interface
between
user
and
hardware
.
It
is
responsible
for
the
execution
of
all
the
processes,
Resource
Allocation,
CPU
management,
File
Management
and
many
other
tasks.
The
purpose
of
an
operating
system
is
to
provide
an
environment
in
w

--- chunk 5 | page 2 ---
processes with
the highest response ratio are scheduled. This algorithm
avoids starvation.
Response Ratio = (Waiting Time + Burst time) / Burst
time
7.
Multilevel Queue Scheduling (MLQ)
: According to the
priority of the
process, processes are placed in the different queues.
Generally high priority


--- chunk 10 | page 5 ---
●
Banker's algorithm
is used to avoid deadlock. It is one of the deadlock-avoidance
methods. It is named as Banker's algorithm on the
banking system where a bank
never allocates available cash in such a manner that
it can no longer satisfy the
requirements of all of its customers.
●
Memory Managemen



Take an example you know: a **DBMS notes PDF** that you want a chatbot to answer questions from. The first stage, called **indexing**, prepares that PDF so questions can be answered later. It's done once, before anyone asks anything.

## The starting point
Imagine page 3 of your PDF contains this:

> **Keys in DBMS.** A primary key uniquely identifies each row in a table. It cannot be null. A table can have only one primary key.
>
> A foreign key is a column in one table that refers to the primary key of another table. It is used to link tables and maintain referential integrity.
>
> **Normalization.** Normalization is the process of organising data to reduce redundancy. 1NF requires atomic values. 2NF removes partial dependency.

A computer can't search this by meaning. It's just a long string. The four steps turn it into something searchable.

## Step 1. Load
A loader reads the file and wraps each page in a `Document`:

```
Document(
  page_content = "Keys in DBMS. A primary key uniquely identifies...",
  metadata = {"source": "dbms_notes.pdf", "page": 3}
)
```
Now your program has the text and a record of where it came from.

## Step 2. Split into chunks
The page mixes three topics. If a student asks "What is a foreign key?", handing the model the whole page, or the whole book, adds noise. So the splitter cuts the text into small, overlapping pieces:

| Chunk | Content (shortened) |
|---|---|
| Chunk 1 | "Keys in DBMS. A primary key uniquely identifies each row... only one primary key." |
| Chunk 2 | "...only one primary key. A foreign key is a column in one table that refers to the primary key of another table..." |
| Chunk 3 | "...maintain referential integrity. Normalization. Normalization is the process of organising data to reduce redundancy..." |

Notice that the end of one chunk is repeated at the start of the next. That's the **overlap**, and it stops an idea from being cut in half at a boundary. Every chunk keeps its metadata (`page: 3`), so the answer can later say where it came from.

## Step 3. Embed
Each chunk is passed to an embedding model, which turns the text into a list of numbers that captures its meaning:

```
Chunk 2 → [0.12, -0.80, 0.33, 0.05, ...]   (hundreds or thousands of numbers)
```
Chunks with similar meaning get similar numbers. "A foreign key links tables" and "a column that references another table's primary key" end up close together, even though they share few words. That's why RAG can find the right chunk without exact keyword matching.

## Step 4. Store
The chunks and their number lists are saved in a **vector database** such as Chroma. It's like a library where books are shelved by topic rather than by title. The indexing stage is now finished.

## Why it matters when someone asks a question
Later the student asks "What is a foreign key?":
1. The question is embedded into numbers the same way.
2. The database finds the closest chunks, which here would be Chunk 2.
3. Only those chunks go to the LLM, which answers from your notes.

If Step 2 was done badly, the answer suffers. For example, with a chunk size so large that foreign keys and normalization sit in the same chunk, retrieval would bring back extra unrelated text. With chunks so small that "A foreign key is a column" is separated from "that refers to another table's primary key", the definition is broken apart.

## See it yourself in Colab
This tiny example lets you watch the splitter work. The sizes are deliberately small so the cuts are visible:

```python
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """Keys in DBMS. A primary key uniquely identifies each row in a table. It cannot be null. A table can have only one primary key.

A foreign key is a column in one table that refers to the primary key of another table. It is used to link tables and maintain referential integrity.

Normalization. Normalization is the process of organising data to reduce redundancy. 1NF requires atomic values. 2NF removes partial dependency."""

doc = Document(page_content=text, metadata={"source": "dbms_notes.pdf", "page": 3})

splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
chunks = splitter.split_documents([doc])

for i, c in enumerate(chunks):
    print(f"--- chunk {i} ---")
    print(c.page_content)
    print(c.metadata)
```
The exact cut points depend on the splitter, so your output may differ slightly from my table. Look for where each chunk starts and ends, and whether any idea got split badly.

**Task:** change `chunk_size` to 60, then to 400, rerun it, and write down which setting would give the best answer to "What is a foreign key?" and why.

## Summary in one line
**Indexing turns your documents into small, labelled, searchable pieces: Load (read) → Split (cut) → Embed (numbers) → Store (database).**



In [18]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_documents(path):
    """Load one PDF/TXT file, or every PDF/TXT file in a folder."""
    path = Path(path)
    files = [path] if path.is_file() else sorted(path.glob("*"))

    docs = []
    for f in files:
        if f.suffix.lower() == ".pdf":
            docs += PyPDFLoader(str(f)).load()
        elif f.suffix.lower() == ".txt":
            docs += TextLoader(str(f), encoding="utf-8").load()
    return docs


def split_documents(docs, chunk_size=1000, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    return splitter.split_documents(docs)


# --- use it ---
docs = load_documents("/content/notes.pdf")   # a file, or a folder like "/content/my_docs"
chunks = split_documents(docs)

print(len(docs), "documents ->", len(chunks), "chunks")
print(chunks[0].page_content[:300])
print(chunks[0].metadata)

0 documents -> 0 chunks


IndexError: list index out of range

# Embeddings and Vector Integration

In [19]:
!pip install -q langchain-chroma langchain-google-genai

**See what an embedding is**

In [20]:
import os
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vector = embeddings.embed_query("A foreign key links two tables.")
print(len(vector))
print(vector[:5])

3072
[-0.024027482, 0.0039357357, 0.029151626, -0.052619632, -0.008030655]


The embedding model turns text into a list of numbers. len(vector) shows how many numbers describe one piece of text, and the first five are just a sample. The model name may be outdated by now, so if you get "model not found", check Google AI Studio's docs for the current embedding model name.

In [21]:
import numpy as np

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

q  = embeddings.embed_query("What is a foreign key?")
s1 = embeddings.embed_query("A foreign key is a column that refers to the primary key of another table.")
s2 = embeddings.embed_query("Pizza is baked in a very hot oven.")

print("related  :", cosine(q, s1))
print("unrelated:", cosine(q, s2))

related  : 0.801563129975516
unrelated: 0.4947269651817265


In [22]:
import os, shutil
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

DB_DIR = "/content/chroma_db"

docs = load_documents("/content/Operating System Notes.pdf")   # a file, or a folder
chunks = split_documents(docs)

print(len(docs), "documents ->", len(chunks), "chunks")
def build_vectorstore(chunks, embeddings, db_dir=DB_DIR):
    """Embed all chunks and save them in a Chroma database."""
    shutil.rmtree(db_dir, ignore_errors=True)   # start fresh, avoids duplicates
    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=db_dir,
    )


def load_vectorstore(embeddings, db_dir=DB_DIR):
    """Reopen a saved database without re-embedding."""
    return Chroma(persist_directory=db_dir, embedding_function=embeddings)


vectorstore = build_vectorstore(chunks, embeddings)
print("Stored chunks:", vectorstore._collection.count())

12 documents -> 25 chunks
Stored chunks: 25
